# Experiment Analysis

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path
from datetime import datetime as dt

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

from matplotlib.figure import Figure

import panel as pn

from enderleaf.const import (
    ImageMergeMode,
    ImageMergeMethod,
    TIME_FORMAT,
    PRECISE_TIME_FORMAT,
    DEFAULT_DATETIME_FORMAT,
    COLOR_SPACES,
)
from enderleaf.draw import image_grid, concat_tile_resize, plot_images_with_histograms
from enderleaf.tools import (
    read_dataframe,
    write_dataframe,
    format_datetime,
    ensure_folder,
)
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    merge_images,
    merge_images_channels,
    match_previous_rotation,
    get_circles,
    get_channels,
    get_channel,
    equalize_hist,
)
from enderleaf.draw import draw_circles

In [ ]:
pn.extension("ipywidgets")

## Constants

In [ ]:
EXP = "Exp26DM09"
INOC = "I1"
# PLATE = 0
MONTH = 6
DAY = 1

PATH_TO_DATA = Path(".").joinpath("output", "job_data", EXP, INOC)
PATH_TO_IMAGES = Path(".").joinpath("output", "images", EXP, INOC)
PATH_TO_PPIMAGES = Path(".").joinpath("output", "pre_processed", EXP, INOC)
MAX_CIRCLES = 3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = (
    pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")])
    .sort_values(["plate", "row", "col"])
    .dropna(subset="north")
)
df = df[df.exp == EXP]
df = df[df.inoc == INOC]
# df = df[df.plate == PLATE]
df = df[df.month == MONTH]
df = df[df.day == DAY]
df["card_count"] = df[["north", "east", "west", "south"]].astype(int).sum(axis=1)
df["file_ok"] = df["file_name"].apply(lambda x: PATH_TO_IMAGES.joinpath(x).is_file())
print(df[df.file_ok == False].shape)
df = df[df.file_ok == True]
df["file_size"] = df["file_name"].apply(lambda x: PATH_TO_IMAGES.joinpath(x).stat().st_size)
# df = df[df.job_ts != 20260515164717]
df["leaf_id"] = df.plate.astype(str) + df.row.astype(str) + df.col.astype(str)
df

In [ ]:
df.groupby("plate").count()

## Select Cycle ID

In [ ]:
sel_plate = pn.widgets.Select(
    name="Plate",
    options=list(df.plate.unique()),
    sizing_mode="scale_width",
    value="P04",
)
sel_row = pn.widgets.Select(
    name="Row", options=list(df.row.unique()), sizing_mode="scale_width", value=7
)
sel_col = pn.widgets.Select(
    name="Col", options=list(df.col.unique()), sizing_mode="scale_width", value="G"
)

sel_color_space = pn.widgets.Select(
    name="Color space",
    options=["rgb", "hsv", "lab", "yuv", "ycrcb"],
    sizing_mode="scale_width",
    value="rgb",
)
sel_channel_1 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][0],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_2 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][1],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_3 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][2],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_merge_method = pn.widgets.Select(
    name="Merge method",
    options={
        k.name: k
        for k in [
            ImageMergeMethod.RGB,
            ImageMergeMethod.HSV,
            ImageMergeMethod.LAB,
            ImageMergeMethod.YUV,
            ImageMergeMethod.YCrCb,
        ]
    },
    sizing_mode="scale_width",
)
bt_random = pn.widgets.Button(name="Random Disc")

img_out = pn.pane.Matplotlib(sizing_mode="scale_width")


updating = False


def filter_df() -> pd.DataFrame:
    return df[
        (df.plate == sel_plate.value)
        & (df.col == sel_col.value)
        & (df.row == sel_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]
        sel_plate.value = row.plate
        sel_row.value = row.row
    finally:
        updating = False
    sel_col.value = row.col


bt_random.on_click(on_random)


@pn.depends(sel_merge_method.param.value, watch=True)
def on_merge_method_changed(mm):
    global updating
    updating = True
    try:
        sel_color_space.value = mm.value[0]
        sel_channel_1.value = mm.value[1][0]
        sel_channel_2.value = mm.value[1][1]
    finally:
        updating = False
    sel_channel_3.value = mm.value[1][2]


@pn.depends(sel_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_channel_1.name = COLOR_SPACES[cs][0]
    sel_channel_2.name = COLOR_SPACES[cs][1]
    sel_channel_3.name = COLOR_SPACES[cs][2]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_plate,
            sel_row,
            sel_col,
            sel_color_space,
            sel_channel_1,
            sel_channel_2,
            sel_channel_3,
        ]
    ],
    watch=True,
)
def on_ld_changed(plate, row, col, cs, cn1, cn2, cn3):
    if updating is True:
        return
    df_ld = filter_df()
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    if len(df_ld.card_count.unique()) > 1:
        merged_images = []
        card_counts = []
        for card_count in df_ld.card_count.unique():
            image_list = [
                crop_image(load(row[1]), crop_data)
                for row in df_ld[df_ld.card_count == card_count].iterrows()
            ]
            merged_images.append(
                merge_images_channels(
                    image_list=image_list, color_space=cs, merge_modes=(cn1, cn2, cn3)
                )
            )
            card_counts.append(f"{card_count} {len(image_list)}")
        img_out.object = plot_images_with_histograms(
            images=merged_images, color_spaces=[cs, "rgb"], titles=card_counts
        )
    else:
        image_list = [crop_image(load(row[1]), crop_data) for row in df_ld.iterrows()]
        image_list.append(
            merge_images_channels(
                image_list=image_list, color_space=cs, merge_modes=(cn1, cn2, cn3)
            )
        )
        img_out.object = plot_images_with_histograms(
            images=image_list, color_spaces=[cs, "rgb"]
        )


on_ld_changed(
    sel_plate.value,
    sel_row.value,
    sel_col.value,
    sel_color_space.value,
    sel_channel_1.value,
    sel_channel_2.value,
    sel_channel_3.value,
)

pn.Column(
    pn.Row(
        sel_plate,
        sel_row,
        sel_col,
        bt_random,
        sel_merge_method,
        sel_color_space,
        sel_channel_1,
        sel_channel_2,
        sel_channel_3,
    ),
    pn.Row(img_out),
)

In [ ]:
sel_cmp_plate = pn.widgets.Select(
    name="Plate",
    options=list(df.plate.unique()),
    sizing_mode="scale_width",
    value="P04",
)
sel_cmp_row = pn.widgets.Select(
    name="Row", options=list(df.row.unique()), sizing_mode="scale_width", value=7
)
sel_cmp_col = pn.widgets.Select(
    name="Col", options=list(df.col.unique()), sizing_mode="scale_width", value="G"
)

sel_cmp_color_space = pn.widgets.Select(
    name="Color space",
    options=["rgb", "hsv", "lab", "yuv", "ycrcb"],
    sizing_mode="scale_width",
    value="rgb",
)
sel_cmp_channel_1 = pn.widgets.Select(
    name=COLOR_SPACES[sel_cmp_color_space.value][0],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_cmp_channel_2 = pn.widgets.Select(
    name=COLOR_SPACES[sel_cmp_color_space.value][1],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_cmp_channel_3 = pn.widgets.Select(
    name=COLOR_SPACES[sel_cmp_color_space.value][2],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_cmp_merge_method = pn.widgets.Select(
    name="Merge method",
    options={
        k.name: k
        for k in [
            ImageMergeMethod.RGB,
            ImageMergeMethod.HSV,
            ImageMergeMethod.LAB,
            ImageMergeMethod.YUV,
            ImageMergeMethod.YCrCb,
        ]
    },
    sizing_mode="scale_width",
)
bt_cmp_random = pn.widgets.Button(name="Random Disc")

img_cmp_out = pn.pane.Matplotlib(sizing_mode="scale_width")


updating = False


def filter_df() -> pd.DataFrame:
    return df[
        (df.plate == sel_cmp_plate.value)
        & (df.col == sel_cmp_col.value)
        & (df.row == sel_cmp_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]
        sel_cmp_plate.value = row.plate
        sel_cmp_row.value = row.row
    finally:
        updating = False
    sel_cmp_col.value = row.col


bt_cmp_random.on_click(on_random)


@pn.depends(sel_cmp_merge_method.param.value, watch=True)
def on_merge_method_changed(mm):
    global updating
    updating = True
    try:
        sel_cmp_color_space.value = mm.value[0]
        sel_cmp_channel_1.value = mm.value[1][0]
        sel_cmp_channel_2.value = mm.value[1][1]
    finally:
        updating = False
    sel_cmp_channel_3.value = mm.value[1][2]


@pn.depends(sel_cmp_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_cmp_channel_1.name = COLOR_SPACES[cs][0]
    sel_cmp_channel_2.name = COLOR_SPACES[cs][1]
    sel_cmp_channel_3.name = COLOR_SPACES[cs][2]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_cmp_plate,
            sel_cmp_row,
            sel_cmp_col,
            sel_cmp_color_space,
            sel_cmp_channel_1,
            sel_cmp_channel_2,
            sel_cmp_channel_3,
        ]
    ],
    watch=True,
)
def on_ld_changed(plate, row, col, cs, cn1, cn2, cn3):
    if updating is True:
        return
    df_ld = filter_df()
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    image_list = [crop_image(load(row[1]), crop_data) for row in df_ld.iterrows()]
    image_list.append(
        merge_images_channels(
            image_list=image_list, color_space=cs, merge_modes=(cn1, cn2, cn3)
        )
    )
    img_rgb = merge_images_channels(
        image_list=image_list,
        color_space="rgb",
        merge_modes=(ImageMergeMode.MIN, ImageMergeMode.MIN, ImageMergeMode.MIN),
    )
    img_cs = merge_images_channels(
        image_list=image_list, color_space=cs, merge_modes=(cn1, cn2, cn3)
    )
    img_cmp_out.object = plot_images_with_histograms(
        images=[img_rgb, img_cs, np.abs(img_cs - img_rgb)],
        color_spaces=["rgb"],
        titles=["rgb", cs, "diff"],
    )


on_ld_changed(
    sel_cmp_plate.value,
    sel_cmp_row.value,
    sel_cmp_col.value,
    sel_cmp_color_space.value,
    sel_cmp_channel_1.value,
    sel_cmp_channel_2.value,
    sel_cmp_channel_3.value,
)

pn.Column(
    pn.Row(
        sel_cmp_plate,
        sel_cmp_row,
        sel_cmp_col,
        bt_cmp_random,
        sel_cmp_merge_method,
        sel_cmp_color_space,
        sel_cmp_channel_1,
        sel_cmp_channel_2,
        sel_cmp_channel_3,
    ),
    pn.Row(img_cmp_out),
)